[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/06_LagEmbeddedFeatures.ipynb)

# Lag-Embedded Feature Matrices

**Module 0 · Lesson 6 of 13 · Student edition**  
**Estimated class time:** 85–100 minutes  
**Source sequence:** Original Day 2  

**Prerequisite:** Lessons 4–5  

## Learning objectives

By the end of this lesson, you should be able to:

- Reframe autoregression as supervised learning.
- Construct univariate and multivariate lag matrices.
- Relate an AR model to linear regression on lagged features.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()
series = df['Log_Diff'].dropna()

# The preceding lesson selected p=12 from the seasonal PACF signature.
p = 12
result = AutoReg(series, lags=p).fit()


---
## Part 5 — Re-framing as a Supervised Learning Problem

### 5.1 The key idea

An AR($p$) model is essentially a **linear regression** where the features are lagged
values of the target:

$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + \cdots + \phi_p x_{t-p} + \varepsilon_t$$

This means we can turn any time series into a tabular dataset that *any* machine
learning model can consume — not just AR. This is the foundation for tree-based
models, MLPs, and other models we'll use later this week.

The transformation looks like this:

| $x_{t-2}$ | $x_{t-1}$ | $x_t$ (target) |
|---|---|---|
| $x_1$ | $x_2$ | $x_3$ |
| $x_2$ | $x_3$ | $x_4$ |
| $x_3$ | $x_4$ | $x_5$ |
| ⋮ | ⋮ | ⋮ |

Each row is one training example. The **lag columns** are features ($X$);
the current value is the **target** ($y$). This is called a **lag-embedded feature matrix**.

### 5.2 Build the lag matrix function

Below is the skeleton of a function that creates a lag-embedded feature matrix
from a univariate time series. **Fill in the missing lines.**

> 💡 **Syntax reminder:** `np.roll(arr, shift)` shifts an array by `shift` positions.
> `pd.DataFrame.shift(k)` shifts a Series by `k` time steps (introduces NaN at the start).


In [ ]:
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series  : array-like, shape (T,)
    n_lags  : int, number of lag features to create

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)   <- feature matrix
    y : np.ndarray, shape (T - n_lags,)           <- target vector
    """
    series = np.array(series)
    T = len(series)
    X, y = [], []

    for t in range(n_lags, T):
        # FILL IN: features are the previous n_lags values
        X.append(???)
        # FILL IN: target is the value at time t
        y.append(???)

    return np.array(X), np.array(y)


In [ ]:
# Test your function on the log-differenced series
X, y = make_lag_matrix(series.values, n_lags=3)

print('Feature matrix X shape:', X.shape)
print('Target vector y shape: ', y.shape)
print()
print('First 5 rows of X (lag features):')
print(X[:5])
print()
print('First 5 targets y:')
print(y[:5])


### 5.3 View as a tidy DataFrame

**Your turn!** Wrap the output of `make_lag_matrix` in a `pd.DataFrame` with
descriptive column names (`lag_1`, `lag_2`, ..., `lag_p`, `target`).


In [ ]:
n_lags = 3
X, y = make_lag_matrix(series.values, n_lags=n_lags)

# FILL IN: create column names ['lag_1', 'lag_2', 'lag_3', 'target']
col_names = [f'lag_{i}' for i in range(???, ???)] + ['???']

lag_df = pd.DataFrame(
    np.column_stack([X, y]),
    columns=col_names
)

print('Shape:', lag_df.shape)
lag_df.head(10)


### ✏️ Written Response 5.3

1. How many rows does the lag matrix have compared to the original series? Why?
2. In plain language, what does each *row* of the lag matrix represent?
3. If you used `n_lags=12` on monthly data, what would that mean conceptually?

> **YOUR ANSWER:**


---
## Part 6 — Multivariate Lag Matrices

The same idea extends to **multiple time series**. For example, you might want to
predict passenger counts using lagged values of *both* passenger counts and some
other variable (e.g., fuel prices, GDP, temperature).

We'll simulate a second series to demonstrate the concept.

### 6.1 Create a synthetic multivariate dataset

This cell is complete — run it.


In [ ]:
# Use log-differenced passengers as series 1
s1 = series.values

# Simulate a correlated second series (e.g., a noisy economic indicator)
np.random.seed(0)
s2 = 0.5 * s1 + np.random.normal(0, 0.02, size=len(s1))

# Pack into a DataFrame
multi_df = pd.DataFrame({'passengers_log_diff': s1, 'indicator': s2})
print('Shape:', multi_df.shape)
multi_df.head()


### 6.2 Build a multivariate lag matrix function

**Your turn!** Fill in the function below. The idea is the same as before, but now
you create lag columns for *every* variable in the DataFrame.

> 💡 **Syntax reminder:** `df.shift(k)` shifts every column of a DataFrame down by
> `k` rows (the first `k` rows become NaN). `df.dropna()` removes rows with any NaN.


In [ ]:
def make_lag_matrix_multi(df, n_lags, target_col):
    """
    Build a lag-embedded feature matrix from a multivariate time series.

    Parameters
    ----------
    df         : pd.DataFrame, shape (T, n_variables)
    n_lags     : int, number of lags per variable
    target_col : str, column name of the variable to forecast

    Returns
    -------
    X : pd.DataFrame of lag features
    y : pd.Series of targets
    """
    lagged_frames = []

    for lag in range(1, n_lags + 1):
        # FILL IN: shift the entire DataFrame by `lag` steps
        shifted = df.shift(???)
        # FILL IN: rename columns to indicate the lag, e.g. 'passengers_log_diff_lag1'
        shifted.columns = [f'{col}_lag{lag}' for col in df.columns]
        lagged_frames.append(???)

    # Combine all lagged frames side by side
    feature_df = pd.concat(lagged_frames, axis=1)

    # FILL IN: the target is the current (unshifted) target column
    target = df[???]

    # Drop rows where any lag is NaN (the first n_lags rows)
    combined = pd.concat([feature_df, target], axis=1).dropna()
    X = combined.drop(columns=[target_col])
    y = combined[target_col]

    return X, y


In [ ]:
# Test the multivariate function
X_multi, y_multi = make_lag_matrix_multi(
    df=multi_df,
    n_lags=3,
    target_col='passengers_log_diff'
)

print('Feature matrix shape:', X_multi.shape)
print('Target vector shape: ', y_multi.shape)
print()
print('Column names:')
print(list(X_multi.columns))
print()
X_multi.head()


### ✏️ Written Response 6.2

1. How many feature columns does the multivariate lag matrix have? How is this computed
   from the number of variables and the number of lags?
2. Why do we drop the first `n_lags` rows after building the lag matrix?
3. In what situation would adding a second (or third) variable's lags be useful?
   Give a real-world example.

> **YOUR ANSWER:**


---
## Part 7 — Putting It All Together

### 7.1 From AR to supervised learning: the connection

The lag matrix you built in Part 5 is *exactly* the input representation needed to
use *any* machine learning model for forecasting. Let's verify that a simple linear
regression on the lag matrix gives the same result as `AutoReg`.

This cell is complete — run it and compare the coefficients.


In [ ]:
from sklearn.linear_model import LinearRegression

# Build the lag matrix with the same order as your AR model above
X_lr, y_lr = make_lag_matrix(series.values, n_lags=p)

lr = LinearRegression(fit_intercept=True)
lr.fit(X_lr, y_lr)

print('LinearRegression coefficients (lag_1, lag_2, ...):')
print(lr.coef_)
print('Intercept:', lr.intercept_)
print()
print('AutoReg coefficients (intercept, lag_1, lag_2, ...):')
print(result.params.values)


### ✏️ Final Written Reflection

Write a **4–6 sentence summary** connecting today's topics as if explaining to a
classmate who missed Day 2. Your summary must address:

- What an AR($p$) model does and what the coefficients mean
- Why the stability condition $|\phi| < 1$ matters
- How you used the PACF to choose the order $p$
- How a lag-embedded feature matrix re-frames forecasting as supervised learning
- Why this re-framing is powerful (what models does it unlock?)

> **YOUR ANSWER:**


---
## Moving Forward

Choose a different time series (your own data or any publicly available dataset) and:

1. Run the `ts_diagnostics()` function from Day 1
2. Use the PACF to select an AR order
3. Fit an `AutoReg` model and check the stability condition
4. Build a lag matrix using `make_lag_matrix`
5. Fit a `LinearRegression` model on the lag matrix and compare coefficients

```python
# Your code here
```


## References / Further Reading

* [Forecasting: Principles and Practice — Chapter 9 (ARIMA)](https://otexts.com/fpp3/arima.html)
* [statsmodels AutoReg documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.ar_model.AutoReg.html)
* [Machine Learning for Time Series Forecasting — Géron](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/)
